# Step 2: Merge + Evaluate

Merges all LoRA adapters (SFT + GRPO) with base Qwen3.5-4B, then runs lm-evaluation-harness.

**Runtime:** H100 GPU
**Input:** `sft_lora_*/` and `grpo_lora_*/` on Drive (from notebook 01)
**Output:** `merged_*/` models + `eval_results/` on Drive
**Time:** Merge ~5 min per model, eval ~30 min per model

In [ ]:
# Install — restart runtime after!
!pip install unsloth lm-eval --upgrade --quiet
print('Done. Now: Runtime → Restart session, then run next cell.')

In [ ]:
# Setup (after restart)
import transformers
from transformers.models.auto.configuration_auto import CONFIG_MAPPING
assert 'qwen3_5' in CONFIG_MAPPING, f'Need transformers>=5.0, got {transformers.__version__}'
print(f'transformers {transformers.__version__} ✅')

from google.colab import drive
drive.mount('/content/drive')

import os, json, glob, subprocess
DRIVE = '/content/drive/MyDrive/distillreasoning'
os.makedirs(f'{DRIVE}/eval_results', exist_ok=True)
print(f'Drive: {DRIVE} ✅')

In [ ]:
# Merge all LoRA adapters with base model
from unsloth import FastLanguageModel
import torch

# Find all LoRA dirs on Drive
lora_dirs = {}
for prefix in ['sft_lora', 'grpo_lora']:
    for teacher in ['glm5', 'kimi', 'combined']:
        lora_dir = f'{DRIVE}/{prefix}_{teacher}'
        if os.path.exists(lora_dir) and any(f.endswith('.safetensors') for f in os.listdir(lora_dir)):
            name = f'{prefix.replace("_lora", "")}_{teacher}'  # sft_glm5, grpo_kimi, etc
            lora_dirs[name] = lora_dir

print(f'Found {len(lora_dirs)} LoRA adapters to merge:')
for name, path in lora_dirs.items():
    print(f'  {name}: {path}')

# Merge each
for name, lora_dir in lora_dirs.items():
    merged_dir = f'{DRIVE}/merged_{name}'
    if os.path.exists(merged_dir) and any(f.endswith('.safetensors') for f in os.listdir(merged_dir)):
        print(f'{name}: already merged')
        continue
    
    print(f'Merging {name}...')
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=lora_dir, max_seq_length=4096, dtype=None, load_in_4bit=False,
    )
    model.save_pretrained_merged(merged_dir, tokenizer, save_method='merged_16bit')
    del model, tokenizer
    torch.cuda.empty_cache()
    print(f'  ✅ Saved: {merged_dir}')

print('\nAll models merged!')

In [ ]:
# Build model list for evaluation
EVAL_MODELS = [
    # (name, path, is_chat)
    ('base-qwen35-4b', 'Qwen/Qwen3.5-4B', True),
    ('llama-3.2-3b', 'meta-llama/Llama-3.2-3B', False),
    ('qwen3-8b', 'Qwen/Qwen3-8B', False),
    # ('gpt-oss-20b', 'openai/gpt-oss-20b', True),  # Uncomment if H100 80GB
]

# Add all merged models
for name in lora_dirs:
    merged_dir = f'{DRIVE}/merged_{name}'
    if os.path.exists(merged_dir):
        EVAL_MODELS.append((name, merged_dir, True))

print(f'{len(EVAL_MODELS)} models to evaluate:')
for name, path, chat in EVAL_MODELS:
    print(f'  {name:25s} chat={chat}')

In [ ]:
# Run lm-evaluation-harness on each model
# Industry standard — same tool used by HuggingFace Open LLM Leaderboard
# ARC/GPQA/MMLU use log-likelihood (no generation needed, no extraction bugs)
# GSM8K/MATH use generation with proper regex/boxed extraction

ALL_TASKS = 'gsm8k_cot,minerva_math,arc_challenge,gpqa_diamond,mmlu_pro'

for name, model_path, is_chat in EVAL_MODELS:
    result_dir = f'{DRIVE}/eval_results/{name}'
    
    # Check if already done
    existing = glob.glob(f'{result_dir}/**/results.json', recursive=True)
    if existing:
        print(f'{name}: already evaluated ✅')
        continue
    
    print(f'\nEvaluating: {name}...')
    cmd = [
        'lm-eval', '--model', 'hf',
        '--model_args', f'pretrained={model_path},dtype=bfloat16,trust_remote_code=True',
        '--tasks', ALL_TASKS,
        '--batch_size', 'auto',
        '--output_path', result_dir,
        '--log_samples',
    ]
    if is_chat:
        cmd.extend(['--apply_chat_template', '--fewshot_as_multiturn'])
    
    print(f'  {" ".join(cmd[:8])}...')
    result = subprocess.run(cmd, capture_output=True, text=True)
    
    if result.returncode == 0:
        print(f'  ✅ Done')
    else:
        print(f'  ❌ Error')
        print(result.stderr[-500:])
    
    # Show last part of output (includes scores)
    print(result.stdout[-1000:])

In [ ]:
# Collect and display results table
all_results = {}

for name, _, _ in EVAL_MODELS:
    result_files = glob.glob(f'{DRIVE}/eval_results/{name}/**/results.json', recursive=True)
    if not result_files:
        continue
    with open(result_files[0]) as f:
        data = json.load(f)
    results = {}
    for task_name, task_data in data.get('results', {}).items():
        for metric in ['exact_match,flexible-extract', 'exact_match,strict-match',
                       'acc_norm,none', 'acc,none']:
            if metric in task_data:
                results[task_name] = round(task_data[metric] * 100, 1)
                break
    all_results[name] = results

# Print table
benchmarks = ['gsm8k_cot', 'minerva_math', 'arc_challenge', 'gpqa_diamond', 'mmlu_pro']
labels = ['GSM8K', 'MATH', 'ARC', 'GPQA', 'MMLU-Pro']

print(f'{"Model":25s}', end='')
for label in labels:
    print(f'{label:>10s}', end='')
print()
print('-' * (25 + 10 * len(labels)))
for name, _, _ in EVAL_MODELS:
    if name not in all_results:
        continue
    print(f'{name:25s}', end='')
    for bench in benchmarks:
        score = all_results[name].get(bench, '—')
        s = f'{score}%' if isinstance(score, (int, float)) else str(score)
        print(f'{s:>10s}', end='')
    print()

with open(f'{DRIVE}/eval_results/comparison.json', 'w') as f:
    json.dump(all_results, f, indent=2)
print(f'\nSaved: {DRIVE}/eval_results/comparison.json')

In [ ]:
# Trick questions — side by side
from unsloth import FastLanguageModel
import torch

TRICKS = [
    ('A farmer has 17 sheep. All but 9 die. How many sheep does the farmer have left?', '9'),
    ('If it takes 5 machines 5 minutes to make 5 widgets, how long would it take 100 machines to make 100 widgets?', '5 minutes'),
    ('A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. How much does the ball cost?', '$0.05'),
]

SYS = 'You are a helpful reasoning assistant. Think through problems step by step.'

for model_name in ['base-qwen35-4b', 'sft_kimi', 'grpo_kimi']:
    path = next((p for n, p, _ in EVAL_MODELS if n == model_name), None)
    if not path:
        print(f'{model_name}: not found, skipping')
        continue
    model, tok = FastLanguageModel.from_pretrained(model_name=path, max_seq_length=2048, load_in_4bit=True)
    FastLanguageModel.for_inference(model)
    print(f'\n=== {model_name} ===')
    for q, expected in TRICKS:
        msgs = [{'role':'system','content':SYS}, {'role':'user','content':q}]
        inp = tok.apply_chat_template(msgs, return_tensors='pt', add_generation_prompt=True).to('cuda')
        out = model.generate(inp, max_new_tokens=512, temperature=0.6, do_sample=True)
        resp = tok.decode(out[0][inp.shape[-1]:], skip_special_tokens=True)
        print(f'  Q: {q[:60]}...')
        print(f'  Expected: {expected}')
        print(f'  Model: {resp[:200]}')
        print()
    del model, tok
    torch.cuda.empty_cache()

print('\n🎉 Eval complete! Run 03_publish.ipynb to push best model to HuggingFace.')